# Comparaison de Performance GPU (A100) : Solver de Clusters vs Baseline CDCL (PySAT)

Ce notebook implémente et compare deux approches de résolution pour le problème 3-SAT sur des instances structurées réelles :
1. **Baseline CDCL** : Résolution via MiniSat/Glucose via la bibliothèque `PySAT` (s'exécutant sur CPU).
2. **Solver de Clusters Hybride Optimisé GPU** : Implémentation complète en **PyTorch** s'exécutant entièrement sur le GPU (A100). Elle utilise la propagation d'étiquettes parallèle pour la phase de gel de Swendsen-Wang, un échantillonnage MCMC parallèle, et une décomposition spectrale sur GPU pour le clustering final.

## 0. Configuration de l'environnement

In [ ]:
# Installation de PySAT
!pip install python-sat

import time
import random
import numpy as np
import torch
from scipy.optimize import linprog
import matplotlib.pyplot as plt

# Vérification du GPU
print("GPU Disponible :", torch.cuda.is_available())
if torch.cuda.is_available():
    print("Nom du GPU :", torch.cuda.get_device_name(0))

## 1. Générateur d'Instances Structurées Réelles (Vérification de Circuits)

Pour simuler un problème industriel réel (et non du 3-SAT aléatoire sans structure), nous générons un problème de **vérification d'équivalence de circuits logiques** (Miter). 
We create two logic circuits almost identical performing operations on inputs, but with a mutation introduced in the second circuit. We then apply the **Tseitin transformation** to obtain a satisfiable CNF formula (the two circuits differ).

In [ ]:
def generate_miter_circuit_3sat(n_inputs=50, n_gates=150):
    """
    Génère une instance de vérification d'équivalence de circuits sous forme de clauses 3-SAT.
    """
    clauses = []
    next_var = n_inputs + 1
    
    # Représentation des portes : (type, out_var, in1, in2)
    gates1 = []
    gates2 = []
    
    # Génération du circuit 1
    current_vars = list(range(1, n_inputs + 1))
    for _ in range(n_gates):
        gtype = random.choice(["AND", "OR", "XOR"])
        in1 = random.choice(current_vars)
        in2 = random.choice(current_vars)
        out = next_var
        next_var += 1
        gates1.append((gtype, out, in1, in2))
        current_vars.append(out)
    output1 = current_vars[-1]
    
    # Génération du circuit 2 (identique au circuit 1 mais avec une mutation sur une porte)
    # Cela introduit une différence pour rendre le miter SAT (les circuits ne sont pas équivalents)
    gate_to_mutate = random.randint(0, n_gates - 1)
    var_offset = next_var - (n_inputs + 1)
    
    for i, (gtype, out, in1, in2) in enumerate(gates1):
        out2 = out + var_offset
        in1_2 = in1 if in1 <= n_inputs else in1 + var_offset
        in2_2 = in2 if in2 <= n_inputs else in2 + var_offset
        
        if i == gate_to_mutate:
            # Mutation de la porte
            new_type = "OR" if gtype == "AND" else "AND"
            gates2.append((new_type, out2, in1_2, in2_2))
        else: 
            gates2.append((gtype, out2, in1_2, in2_2))
            
    output2 = current_vars[-1] + var_offset
    next_var = output2 + 1
    
    # Encodage de Tseitin pour le Circuit 1
    for gtype, out, in1, in2 in gates1:
        if gtype == "AND":
            # out = in1 AND in2 <=> (in1 or not out) and (in2 or not out) and (not in1 or not in2 or out)
            clauses.append([in1, -out])
            clauses.append([in2, -out])
            clauses.append([-in1, -in2, out])
        elif gtype == "OR":
            # out = in1 OR in2 <=> (not in1 or out) and (not in2 or out) and (in1 or in2 or not out)
            clauses.append([-in1, out])
            clauses.append([-in2, out])
            clauses.append([in1, in2, -out])
        elif gtype == "XOR":
            # out = in1 XOR in2
            clauses.append([-in1, -in2, -out])
            clauses.append([in1, in2, -out])
            clauses.append([-in1, in2, out])
            clauses.append([in1, -in2, out])
            
    # Encodage de Tseitin pour le Circuit 2
    for gtype, out, in1, in2 in gates2:
        if gtype == "AND":
            clauses.append([in1, -out])
            clauses.append([in2, -out])
            clauses.append([-in1, -in2, out])
        elif gtype == "OR":
            clauses.append([-in1, out])
            clauses.append([-in2, out])
            clauses.append([in1, in2, -out])
        elif gtype == "XOR":
            clauses.append([-in1, -in2, -out])
            clauses.append([in1, in2, -out])
            clauses.append([-in1, in2, out])
            clauses.append([in1, -in2, out])
            
    # Porte Miter XOR finale sur les sorties des deux circuits
    miter_out = next_var
    clauses.append([-output1, -output2, -miter_out])
    clauses.append([output1, output2, -miter_out])
    clauses.append([-output1, output2, miter_out])
    clauses.append([output1, -output2, miter_out])
    
    # On impose que la sortie du Miter soit Vraie
    clauses.append([miter_out])
    
    # Réduction finale en 3-SAT (pour toutes les clauses de taille différente de 3)
    final_3sat_clauses = []
    var_mapping_counter = miter_out + 1
    
    for c in clauses:
        if len(c) == 1:
            # Unitaire : [l] -> [l, dummy, dummy] avec dummy = False
            dummy1 = var_mapping_counter
            dummy2 = var_mapping_counter + 1
            var_mapping_counter += 2
            final_3sat_clauses.append([c[0], dummy1, dummy2])
            final_3sat_clauses.append([c[0], -dummy1, dummy2])
            final_3sat_clauses.append([c[0], dummy1, -dummy2])
            final_3sat_clauses.append([c[0], -dummy1, -dummy2])
        elif len(c) == 2:
            # Binaire : [l1, l2] -> [l1, l2, dummy] avec dummy = False
            dummy = var_mapping_counter
            var_mapping_counter += 1
            final_3sat_clauses.append([c[0], c[1], dummy])
            final_3sat_clauses.append([c[0], c[1], -dummy])
        elif len(c) == 3:
            final_3sat_clauses.append(c)
        else:
            # Plus de 3 littéraux (découpage standard)
            curr_clause = c
            while len(curr_clause) > 3:
                aux = var_mapping_counter
                var_mapping_counter += 1
                final_3sat_clauses.append([curr_clause[0], curr_clause[1], aux])
                curr_clause = [-aux] + curr_clause[2:]
            final_3sat_clauses.append(curr_clause)
            
    num_vars = var_mapping_counter - 1
    return num_vars, final_3sat_clauses

num_vars, clauses_3sat = generate_miter_circuit_3sat(n_inputs=20, n_gates=80)
print(f"Instance générée : {num_vars} variables, {len(clauses_3sat)} clauses 3-SAT.")

## 2. Élimination des Littéraux Purs (Pré-traitement)

Avant de lancer les algorithmes, nous appliquons récursivement l'élimination des littéraux purs.

In [ ]:
def pure_literal_elimination(num_vars, clauses):
    """
    Élimine les variables n'apparaissant qu'avec une seule polarité.
    """
    fixed_literals = {}
    active_clauses = [list(c) for c in clauses]
    active_vars = set(range(1, num_vars + 1))
    
    changed = True
    while changed:
        changed = False
        # Compter les occurrences positives et négatives
        pos_counts = {v: 0 for v in active_vars}
        neg_counts = {v: 0 for v in active_vars}
        
        for c in active_clauses:
            for lit in c:
                var = abs(lit)
                if var in active_vars:
                    if lit > 0:
                        pos_counts[var] += 1
                    else:
                        neg_counts[var] += 1
                        
        pure_vars = []
        for var in list(active_vars):
            pos = pos_counts[var]
            neg = neg_counts[var]
            if pos > 0 and neg == 0:
                pure_vars.append((var, 1))
            elif neg > 0 and pos == 0:
                pure_vars.append((var, -1))
            elif pos == 0 and neg == 0:
                # Variable orpheline
                pure_vars.append((var, 1))
                
        if pure_vars:
            changed = True
            for var, val in pure_vars:
                fixed_literals[var] = val
                active_vars.remove(var)
            
            # Filtrer les clauses résolues par ces littéraux purs
            new_active_clauses = []
            for c in active_clauses:
                satisfied = False
                for lit in c:
                    var = abs(lit)
                    val = 1 if lit > 0 else -1
                    if var in fixed_literals and fixed_literals[var] == val:
                        satisfied = True
                        break
                if not satisfied:
                    # On garde la clause mais on retire les littéraux faussés (normalement aucun car ils sont purs)
                    new_clause = [lit for lit in c if abs(lit) in active_vars]
                    new_active_clauses.append(new_clause)
            active_clauses = new_active_clauses
            
    return active_vars, active_clauses, fixed_literals

active_vars, active_clauses, fixed_lits = pure_literal_elimination(num_vars, clauses_3sat)
print(f"Après pré-traitement : {len(active_vars)} variables actives, {len(active_clauses)} clauses restantes.")

## 3. Double Transfert d'Énergie (Côté CPU pour LP)

Nous formulons les poids d'arêtes initiaux $W_{ij}$, puis nous résolvons le programme linéaire pour allouer les poids aux triangles isotropes non frustrés.

In [ ]:
def perform_double_energy_transfer(active_vars, active_clauses, u=1.0):
    """
    Calcule les poids d'arêtes initiaux, résout le programme d'optimisation linéaire,
    et retourne les structures prêtes pour le transfert vers le GPU.
    """
    var_list = sorted(list(active_vars))
    var_to_idx = {v: i for i, v in enumerate(var_list)}
    N_red = len(var_list)
    
    # 1. Calcul des poids d'arêtes initiaux W_ij
    W_init = {}
    for c in active_clauses:
        if len(c) != 3:
            continue
        i1, i2, i3 = abs(c[0]), abs(c[1]), abs(c[2])
        idx1, idx2, idx3 = var_to_idx[i1], var_to_idx[i2], var_to_idx[i3]
        pols = [1 if x > 0 else -1 for x in c]
        
        pairs = [(idx1, idx2, pols[0]*pols[1]), 
                 (idx2, idx3, pols[1]*pols[2]), 
                 (idx1, idx3, pols[0]*pols[2])]
        
        for u_idx, v_idx, sign in pairs:
            edge = (min(u_idx, v_idx), max(u_idx, v_idx))
            contrib = -sign * (u / 2.0)
            W_init[edge] = W_init.get(edge, 0.0) + contrib

    # Filtrer les arêtes de poids nul
    W_init = {k: v for k, v in W_init.items() if abs(v) > 1e-6}
    edges_list = sorted(list(W_init.keys()))
    edge_to_idx = {e: i for i, e in enumerate(edges_list)}
    
    # Trouver tous les triangles formés par les arêtes actives
    # Un triangle est défini par 3 sommets {a, b, c} connectés
    triangles = []
    adjacency = {i: set() for i in range(N_red)}
    for u_idx, v_idx in edges_list:
        adjacency[u_idx].add(v_idx)
        adjacency[v_idx].add(u_idx)
        
    for a in range(N_red):
        for b in adjacency[a]:
            if b > a:
                for c in adjacency[b]:
                    if c > b and c in adjacency[a]:
                        # Vérification de cohérence des signes de la structure
                        # Le produit des signes des arêtes doit être +1 pour T_iso
                        s1 = np.sign(W_init[(a, b)])
                        s2 = np.sign(W_init[(b, c)])
                        s3 = np.sign(W_init[(a, c)])
                        prod_signs = s1 * s2 * s3
                        triangles.append((a, b, c, prod_signs))
                        
    # 2. Programme Linéaire pour maximiser le transfert vers les triangles
    n_edges = len(edges_list)
    n_triangles = len(triangles)
    
    if n_triangles > 0 and n_edges > 0:
        # Variables : omega_t >= 0 pour chaque triangle t
        c_lp = -np.ones(n_triangles) # Maximiser la somme des omega_t
        
        # Contraintes : pour chaque arête e, la somme des omega_t <= |W_e|
        A_ub = np.zeros((n_edges, n_triangles))
        b_ub = np.zeros(n_edges)
        
        for t_idx, (a, b, c, _) in enumerate(triangles):
            # Trouver les 3 arêtes du triangle
            e1 = edge_to_idx[(a, b)]
            e2 = edge_to_idx[(b, c)]
            e3 = edge_to_idx[(a, c)]
            A_ub[e1, t_idx] = 1.0
            A_ub[e2, t_idx] = 1.0
            A_ub[e3, t_idx] = 1.0
            
        for e_idx, edge in enumerate(edges_list):
            b_ub[e_idx] = abs(W_init[edge])
            
        # Résolution du PL
        res = linprog(c_lp, A_ub=A_ub, b_ub=b_ub, bounds=(0, None), method='highs')
        omega = res.x if res.success else np.zeros(n_triangles)
    else:
        omega = np.zeros(n_triangles)
        
    # 3. Calcul des poids résiduels
    W_res = W_init.copy()
    triangles_nf = [] # Non-frustrés (produit = +1)
    triangles_f = []  # Frustrés (produit = -1)
    
    for t_idx, (a, b, c, prod_signs) in enumerate(triangles):
        w_t = omega[t_idx]
        if w_t > 1e-6:
            # Soustraire du poids des arêtes
            for u_idx, v_idx in [(a, b), (b, c), (a, c)]:
                edge = (u_idx, v_idx)
                W_res[edge] -= np.sign(W_res[edge]) * w_t
                
            # Assigner les signes des arêtes constitutives pour KaTeX/échantillonnage
            s1 = np.sign(W_init[(a, b)])
            s2 = np.sign(W_init[(b, c)])
            s3 = np.sign(W_init[(a, c)])
            
            if prod_signs > 0:
                triangles_nf.append([a, b, c, w_t, s1, s2, s3])
            else:
                triangles_f.append([a, b, c, w_t, s1, s2, s3])
                
    # Garder uniquement les arêtes résiduelles significatives
    W_res = {k: v for k, v in W_res.items() if abs(v) > 1e-6}
    
    return N_red, var_to_idx, W_res, triangles_nf, triangles_f

N_red, var_to_idx, W_res, triangles_nf, triangles_f = perform_double_energy_transfer(active_vars, active_clauses)
print(f"Double Transfert : {len(W_res)} arêtes résiduelles, {len(triangles_nf)} triangles non-frustrés, {len(triangles_f)} triangles frustrés.")

## 4. Implémentation du Solver de Clusters Hybride sur GPU (PyTorch CUDA)

Nous écrivons la dynamique de Markov hybride complète optimisée pour GPU. La phase de gel (percolation) utilise une **propagation d'étiquettes parallèle branchless** qui s'exécute directement en CUDA via PyTorch.

In [ ]:
class GPUPartialClusterDynamicsSolver:
    def __init__(self, N_red, var_to_idx, W_res, triangles_nf, triangles_f, active_clauses, u=1.0):
        self.N_red = N_red
        self.var_to_idx = var_to_idx
        self.u = u
        
        # Transfert des clauses d'énergie orientée vers GPU
        clause_vars = []
        clause_pols = []
        for c in active_clauses:
            if len(c) == 3:
                clause_vars.append([var_to_idx[abs(lit)] for lit in c])
                clause_pols.append([1 if lit > 0 else -1 for lit in c])
                
        self.clause_vars = torch.tensor(clause_vars, dtype=torch.long, device='cuda')
        self.clause_pols = torch.tensor(clause_pols, dtype=torch.float32, device='cuda')
        
        # Transfert des arêtes résiduelles vers GPU
        res_src, res_dst, res_weights = [], [], []
        for (u_node, v_node), weight in W_res.items():
            res_src.append(u_node)
            res_dst.append(v_node)
            res_weights.append(weight)
            
        self.res_src = torch.tensor(res_src, dtype=torch.long, device='cuda')
        self.res_dst = torch.tensor(res_dst, dtype=torch.long, device='cuda')
        self.res_weights = torch.tensor(res_weights, dtype=torch.float32, device='cuda')
        
        # Transfert des triangles non-frustrés (NF) vers GPU
        if triangles_nf:
            nf_arr = np.array(triangles_nf)
            self.nf_vertices = torch.tensor(nf_arr[:, :3], dtype=torch.long, device='cuda')
            self.nf_weights = torch.tensor(nf_arr[:, 3], dtype=torch.float32, device='cuda')
            self.nf_signs = torch.tensor(nf_arr[:, 4:7], dtype=torch.float32, device='cuda')
        else:
            self.nf_vertices = torch.zeros((0, 3), dtype=torch.long, device='cuda')
            self.nf_weights = torch.zeros(0, dtype=torch.float32, device='cuda')
            self.nf_signs = torch.zeros((0, 3), dtype=torch.float32, device='cuda')
            
        # Transfert des triangles frustrés (F) vers GPU
        if triangles_f:
            f_arr = np.array(triangles_f)
            self.f_vertices = torch.tensor(f_arr[:, :3], dtype=torch.long, device='cuda')
            self.f_weights = torch.tensor(f_arr[:, 3], dtype=torch.float32, device='cuda')
            self.f_signs = torch.tensor(f_arr[:, 4:7], dtype=torch.float32, device='cuda')
        else:
            self.f_vertices = torch.zeros((0, 3), dtype=torch.long, device='cuda')
            self.f_weights = torch.zeros(0, dtype=torch.float32, device='cuda')
            self.f_signs = torch.zeros((0, 3), dtype=torch.float32, device='cuda')
            
    def compute_oriented_energy(self, spins):
        """
        Calcul vectoriel sur GPU de la partie orientée de l'énergie.
        """
        # Evaluates: U_ori = u * sum(1 - 1(L1 = L2 = L3 = +1))
        lit_vals = self.clause_pols * spins[self.clause_vars]
        satisfied_clause = (lit_vals == 1.0).all(dim=1)
        unsatisfied_count = (~satisfied_clause).sum()
        return self.u * unsatisfied_count
        
    def run_mcmc(self, steps=1000, burn_in=200):
        """
        Exécute la dynamique hybride et accumule les corrélations sur GPU.
        """
        # Initialisation aléatoire des spins sur GPU
        spins = torch.randint(0, 2, (self.N_red,), dtype=torch.float32, device='cuda') * 2 - 1
        
        sum_spins = torch.zeros(self.N_red, dtype=torch.float64, device='cuda')
        sum_outer = torch.zeros((self.N_red, self.N_red), dtype=torch.float64, device='cuda')
        sample_count = 0
        
        # Variables de cache pour la propagation d'étiquettes
        t_nf_count = self.nf_vertices.shape[0]
        t_f_count = self.f_vertices.shape[0]
        res_count = self.res_src.shape[0]
        
        for step in range(steps):
            frozen_edges_src = []
            frozen_edges_dst = []
            
            # 1. Gel des arêtes résiduelles
            if res_count > 0:
                satisfied_res = (spins[self.res_src] * spins[self.res_dst] * self.res_weights.sign()) > 0
                probs_res = 1.0 - torch.exp(-torch.abs(self.res_weights))
                freeze_res = satisfied_res & (torch.rand(res_count, device='cuda') < probs_res)
                
                frozen_edges_src.append(self.res_src[freeze_res])
                frozen_edges_dst.append(self.res_dst[freeze_res])
                
            # 2. Gel des triangles non-frustrés (NF)
            if t_nf_count > 0:
                e0 = (spins[self.nf_vertices[:, 0]] * spins[self.nf_vertices[:, 1]] * self.nf_signs[:, 0]) > 0
                e1 = (spins[self.nf_vertices[:, 1]] * spins[self.nf_vertices[:, 2]] * self.nf_signs[:, 1]) > 0
                e2 = (spins[self.nf_vertices[:, 0]] * spins[self.nf_vertices[:, 2]] * self.nf_signs[:, 2]) > 0
                
                satisfied_tri = e0 & e1 & e2
                probs_nf = 1.0 - torch.exp(-2.0 * self.nf_weights)
                freeze_nf = satisfied_tri & (torch.rand(t_nf_count, device='cuda') < probs_nf)
                
                # Si gélé, on gèle les 3 arêtes du triangle
                u_nodes = self.nf_vertices[freeze_nf]
                for u_n, v_n in [(u_nodes[:, 0], u_nodes[:, 1]), (u_nodes[:, 1], u_nodes[:, 2]), (u_nodes[:, 0], u_nodes[:, 2])]:
                    frozen_edges_src.append(u_n)
                    frozen_edges_dst.append(v_n)
                    
            # 3. Gel des triangles frustrés (F)
            if t_f_count > 0:
                e0 = (spins[self.f_vertices[:, 0]] * spins[self.f_vertices[:, 1]] * self.f_signs[:, 0]) > 0
                e1 = (spins[self.f_vertices[:, 1]] * spins[self.f_vertices[:, 2]] * self.f_signs[:, 1]) > 0
                e2 = (spins[self.f_vertices[:, 0]] * spins[self.f_vertices[:, 2]] * self.f_signs[:, 2]) > 0
                
                case = e0.to(torch.int32) * 1 + e1.to(torch.int32) * 2 + e2.to(torch.int32) * 4
                
                # Mapping des arêtes satisfaites
                sat_edge1 = torch.where(case == 6, 1, 0)
                sat_edge2 = torch.where(case == 3, 1, 2)
                
                r = torch.rand(t_f_count, device='cuda')
                half_prob = 0.5 * (1.0 - torch.exp(-2.0 * self.f_weights))
                
                freeze1 = (r < half_prob) & (case > 0)
                freeze2 = (r >= half_prob) & (r < 2.0 * half_prob) & (case > 0)
                
                # Récupération des nœuds à geler
                edges1_nodes = torch.where((sat_edge1 == 0).unsqueeze(1), self.f_vertices[:, [0, 1]], 
                                           torch.where((sat_edge1 == 1).unsqueeze(1), self.f_vertices[:, [1, 2]], self.f_vertices[:, [0, 2]]))
                
                frozen_edges_src.append(edges1_nodes[freeze1, 0])
                frozen_edges_dst.append(edges1_nodes[freeze1, 1])
                
                edges2_nodes = torch.where((sat_edge2 == 0).unsqueeze(1), self.f_vertices[:, [0, 1]], 
                                           torch.where((sat_edge2 == 1).unsqueeze(1), self.f_vertices[:, [1, 2]], self.f_vertices[:, [0, 2]]))
                
                frozen_edges_src.append(edges2_nodes[freeze2, 0])
                frozen_edges_dst.append(edges2_nodes[freeze2, 1])
                
            # 4. Propagation d'étiquettes parallèle sur GPU (Label Propagation)
            labels = torch.arange(self.N_red, device='cuda')
            
            if len(frozen_edges_src) > 0:
                src = torch.cat(frozen_edges_src)
                dst = torch.cat(frozen_edges_dst)
                
                sym_src = torch.cat([src, dst])
                sym_dst = torch.cat([dst, src])
                
                if sym_src.shape[0] > 0:
                    for _ in range(50):
                        labels_next = torch.scatter_reduce(labels, 0, sym_dst, labels[sym_src], reduce='amax', include_self=True)
                        if torch.equal(labels, labels_next):
                            break
                        labels = labels_next
                        
            # 5. Recoloriage de Metropolis-Hastings (Partie Orientée)
            unique_labels, cluster_indices = torch.unique(labels, return_inverse=True)
            M = unique_labels.shape[0]
            
            flips = (torch.rand(M, device='cuda') > 0.5).float() * 2 - 1
            candidate_spins = spins * flips[cluster_indices]
            
            E_curr = self.compute_oriented_energy(spins)
            E_cand = self.compute_oriented_energy(candidate_spins)
            
            dE = E_cand - E_curr
            alpha = torch.exp(-dE)
            
            if torch.rand(1, device='cuda') < alpha:
                spins = candidate_spins
                
            # 6. Accumulation pour la matrice de corrélation
            if step >= burn_in:
                sum_spins += spins
                sum_outer += torch.outer(spins, spins)
                sample_count += 1
                
        mean_spins = sum_spins / sample_count
        mean_outer = sum_outer / sample_count
        covariance = mean_outer - torch.outer(mean_spins, mean_spins)
        
        return covariance
        
    def perform_signed_spectral_clustering(self, covariance):
        """
        Exécute le clustering spectral signé sur GPU via torch.linalg.eigh.
        """
        D = torch.diag(torch.abs(covariance).sum(dim=1))
        L_signed = D - covariance
        
        eigenvalues, eigenvectors = torch.linalg.eigh(L_signed)
        v_min = eigenvectors[:, 0]
        
        spins_spectral = torch.sign(v_min)
        return spins_spectral

## 5. Fonction d'Évaluation Globale et Recomposition de la Solution

Cette fonction orchestre le pré-traitement, le double transfert, l'échantillonnage MCMC GPU, le clustering spectral et la réincorporation des variables éliminées.

In [ ]:
def evaluate_3sat_energy(spins, clauses):
    """
    Évalue le nombre de clauses non satisfaites pour une configuration donnée (CPU).
    """
    unsat = 0
    for c in clauses:
        satisfied = False
        for lit in c:
            var = abs(lit)
            val = 1 if lit > 0 else -1
            if spins[var] == val:
                satisfied = True
                break
        if not satisfied:
            unsat += 1
    return unsat

def solve_3sat_with_gpu_cluster_dynamics(num_vars, clauses, steps=800, burn_in=200):
    """
    Orchestrateur global pour le solver de clusters GPU.
    """
    t_start = time.time()
    
    # 1. Élimination des littéraux purs
    active_vars, active_clauses, fixed_lits = pure_literal_elimination(num_vars, clauses)
    
    if len(active_vars) == 0:
        full_spins = {v: fixed_lits.get(v, 1) for v in range(1, num_vars + 1)}
        return full_spins, time.time() - t_start
        
    # 2. Double transfert d'énergie
    N_red, var_to_idx, W_res, triangles_nf, triangles_f = perform_double_energy_transfer(active_vars, active_clauses)
    
    # 3. Initialisation du solver GPU
    solver = GPUPartialClusterDynamicsSolver(N_red, var_to_idx, W_res, triangles_nf, triangles_f, active_clauses)
    
    # 4. Échantillonnage MCMC GPU
    covariance = solver.run_mcmc(steps=steps, burn_in=burn_in)
    
    # 5. Clustering Spectral GPU
    spins_red_cuda = solver.perform_signed_spectral_clustering(covariance)
    spins_red = spins_red_cuda.cpu().numpy()
    
    # 6. Recomposition et sélection de la meilleure orientation globale (modulo le signe global)
    cand1 = {}
    cand2 = {}
    
    var_list = sorted(list(active_vars))
    for i, var in enumerate(var_list):
        cand1[var] = int(spins_red[i])
        cand2[var] = -int(spins_red[i])
        
    for var, val in fixed_lits.items():
        cand1[var] = val
        cand2[var] = val
        
    unsat1 = evaluate_3sat_energy(cand1, clauses)
    unsat2 = evaluate_3sat_energy(cand2, clauses)
    
    best_spins = cand1 if unsat1 < unsat2 else cand2
    t_elapsed = time.time() - t_start
    
    return best_spins, t_elapsed

## 6. Baseline CDCL (PySAT MiniSat/Glucose)

Nous exécutons la baseline PySAT sur CPU afin de comparer les performances en temps de calcul et qualité de solution.

In [ ]:
from pysat.solvers import Glucose4

def solve_3sat_with_pysat(num_vars, clauses):
    """
    Résout l'instance 3-SAT via Glucose 4 de PySAT (CPU).
    """
    t_start = time.time()
    with Glucose4(bootstrap_with=clauses) as solver:
        satisfied = solver.solve()
        model = solver.get_model() if satisfied else None
    t_elapsed = time.time() - t_start
    
    if model:
        spins = {abs(lit): (1 if lit > 0 else -1) for lit in model}
        return spins, t_elapsed
    else:
        return None, t_elapsed

## 7. Exécution du Benchmark Comparatif

Nous lançons une série de tests sur des instances de miter de tailles croissantes (nombre de variables) et mesurons le temps de résolution pour les deux approches.

In [ ]:
bench_sizes = [50, 100, 200, 400, 600]
results_gpu_time = []
results_gpu_unsat = []
results_cpu_time = []
results_cpu_unsat = []

print("Lancement du benchmark comparatif...\n")
for size in bench_sizes:
    n_inputs = size // 4
    n_gates = size
    
    num_vars, clauses = generate_miter_circuit_3sat(n_inputs=n_inputs, n_gates=n_gates)
    print(f"Taille cible : {size} variables. Instance générée : {num_vars} variables, {len(clauses)} clauses.")
    
    # 1. Résolution avec le Solver de Clusters GPU
    gpu_spins, gpu_time = solve_3sat_with_gpu_cluster_dynamics(num_vars, clauses, steps=1000, burn_in=300)
    gpu_unsat = evaluate_3sat_energy(gpu_spins, clauses)
    results_gpu_time.append(gpu_time)
    results_gpu_unsat.append(gpu_unsat)
    print(f"  [GPU Cluster] Temps : {gpu_time:.4f} s | Clauses insatisfaites : {gpu_unsat}")
    
    # 2. Résolution avec la Baseline PySAT CPU
    cpu_spins, cpu_time = solve_3sat_with_pysat(num_vars, clauses)
    if cpu_spins:
        cpu_unsat = evaluate_3sat_energy(cpu_spins, clauses)
    else:
        cpu_unsat = -1
    results_cpu_time.append(cpu_time)
    results_cpu_unsat.append(cpu_unsat)
    print(f"  [PySAT CDCL]  Temps : {cpu_time:.4f} s | Clauses insatisfaites : {cpu_unsat}\n")

## 8. Visualisation des Résultats

Nous traçons les graphiques comparatifs de temps de calcul.

In [ ]:
plt.figure(figsize=(10, 5))
plt.plot(bench_sizes, results_gpu_time, 'o-', color='indigo', label='GPU Cluster Solver (PyTorch CUDA)')
plt.plot(bench_sizes, results_cpu_time, 's--', color='darkorange', label='CPU Baseline (PySAT Glucose4)')
plt.xlabel('Taille du problème (Nombre de portes dans le circuit Miter)')
plt.ylabel('Temps d\'exécution (secondes)')
plt.title('Comparaison de Performance : Solver de Clusters GPU vs PySAT CDCL')
plt.grid(True, linestyle=':')
plt.legend()
plt.show()

# Affichage des clauses insatisfaites sous forme de tableau
print("Variables\t| GPU Unsat Clauses\t| PySAT Unsat Clauses")
print("-"*55)
for idx, size in enumerate(bench_sizes):
    print(f"{size}\t\t| {results_gpu_unsat[idx]}\t\t\t| {results_cpu_unsat[idx]}")